In [ ]:
!pip install supervision tqdm

import cv2
import supervision as sv
from google.colab import drive
import numpy as np
import pandas as pd
from tqdm import tqdm

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Vamos a crear el dataset de los robots y luego lo vamos a filtrar y limpiar

## Funcion para importar las detecciones y proyectar al campo canonico

In [ ]:
#FUNCION PARA IMPORTAR LAS DETECCIONES
from pycocotools import mask as mask_utils
import json
import os
import numpy as np

#PARA CARGAR LOS DATOS NUEVAMENTE
def cargar_detecciones_frame(indice_frame:int,ruta_carpeta:str):
    """
    Reconstruye sv.Detections desde el JSON guardado. Igual me la dio Claude
    """
    in_path = os.path.join(ruta_carpeta,f"frame_{indice_frame:08d}.json")
    with open(in_path) as f:
        datos_rle = json.load(f)

    if len(datos_rle) ==0:
        return sv.Detections.empty()
    mascaras,cajas,confianza,clase,tracker_ids = [],[],[],[],[]
    data_acumulado = {}

    for objeto in datos_rle:
        rle = objeto["rle"]
        rle["counts"] = rle["counts"].encode("utf-8") #de vuelta a bytes
        mascaras.append(mask_utils.decode(rle).astype(bool))
        cajas.append(objeto["xyxy"])
        confianza.append(objeto.get("confidence"))
        clase.append(objeto.get("class_id"))
        tracker_ids.append(objeto.get("tracker_id"))

        #DATA
        for clave, valores in objeto.get("data",{}).items():
            data_acumulado.setdefault(clave,[]).append(valores)

    data_final = {clave: np.array(valores) for clave, valores in data_acumulado.items()}

    return sv.Detections(
        xyxy = np.array(cajas,dtype=np.float32),
        mask=np.stack(mascaras),
        confidence = np.array(confianza,dtype=np.float32) if confianza[0] is not None else None,
        class_id=np.array(clase, dtype=int) if clase[0] is not None else None,
        tracker_id=np.array(tracker_ids, dtype=int) if tracker_ids[0] is not None else None,
        data=data_final if data_final else {},
    )


def proyectar_puntos(detecciones,matriz):
    #Esta funcion recibe un objeto de detecciones y los manda al espacio canonico
    #Devueve una lista de puntos canonicos
    #Proyecta solo el CENTRO DE LA CAJA

    cajas = detecciones.xyxy
    lista_x=[]
    lista_y=[]

    for caja in cajas:
        x1=caja[0];y1=caja[1]
        x2=caja[2];y2=caja[3]
        x = (x1+x2)/2
        y = (y1+y2)/2

        punto = np.float32([x,y])
        proyeccion = cv2.perspectiveTransform(punto.reshape(1,1,2),matriz)
        lista_x.append(int(proyeccion[0][0][0]))
        lista_y.append(int(proyeccion[0][0][1]))

        #print()

    return lista_x,lista_y


## Recorrer el video frame por frame

In [ ]:
CARPETA_FRAMES = "/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo"
RUTA_OBJETIVO = "/content/drive/MyDrive/PROYECTO SAM 3/datasets/datos_robots_video_completo/v2-datos_robots_video_completo.csv"
RUTA_VIDEO = "/content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV"


#Matriz de transformacion para obtener los puntos canonicos
H = np.array([
    [1.14741274, -0.180942021, 401.053893],
    [0.136897422, 1.14115349, -258.833614],
    [-0.000017149164, -0.0000341202393, 1.0]
])


In [ ]:
#INICIALIZAR EL DICCIONARIO PARA GUARDAR LOS DATOS
#Son las 3:43 de la manana del martes 16 y apenas descubro que supervision tiene
#una funcion para exportar a csv...
#pero pues ya ni modo mejor termino lo mio. Chance y el otro estaba mas feo

datos = {
    "frame": [],
    "tracker_id": [],
    "class_name": [],
    "x_canon": [],
    "y_canon": [],
    "area_caja":[],
    "area_mascara":[],
    "ancho_caja":[],
    "alto_caja":[],
}


#1. Inicializar el capturador de video
capturador = cv2.VideoCapture(RUTA_VIDEO)
frames_totales = int(capturador.get(cv2.CAP_PROP_FRAME_COUNT))

#2. Recorrer todos y cada uno de los frames

with tqdm(total=frames_totales,desc="Generando datos de robots...",unit="Frame") as pbar:

    indice_frame = 1

    while capturador.isOpened():
        ret,frame = capturador.read()
        if not ret: break

        #Cargar las detecciones con el try, si sale algun error se salta al siguiente frame
        try:
            detecciones = cargar_detecciones_frame(indice_frame,CARPETA_FRAMES)
            #print(detecciones.data)
            n = len(detecciones)
            lista_x_canon, lista_y_canon = proyectar_puntos(detecciones,H)
            escala = 8


            lista_x_real = [x/escala for x in lista_x_canon]
            lista_y_real = [y/escala for y in lista_y_canon]

            area_caja = detecciones.box_area.tolist()
            area_mascara = detecciones.area.tolist()
            ancho_caja = (detecciones.xyxy[:,2]-detecciones.xyxy[:,0]).tolist()
            alto_caja = (detecciones.xyxy[:,3]-detecciones.xyxy[:,1]).tolist()

            #print("\n")
            #print("Area caja: ",area_caja)
            #print("Area mascara: ",area_mascara)
            #print("Ancho caja: ",ancho_caja)
            #print("Alto caja: ",alto_caja)

            datos["frame"] = datos["frame"] + [indice_frame]*n
            datos['tracker_id'] = datos["tracker_id"] + detecciones.tracker_id.tolist()
            datos["class_name"] = datos["class_name"] + detecciones.data["class_name"].tolist()
            datos["x_canon"] = datos["x_canon"] + lista_x_real
            datos["y_canon"] = datos["y_canon"] + lista_y_real

            datos["area_caja"] = datos["area_caja"] + area_caja
            datos["area_mascara"] = datos["area_mascara"] + area_mascara
            datos["ancho_caja"] = datos["ancho_caja"] + ancho_caja
            datos["alto_caja"] = datos["alto_caja"] + alto_caja


        except Exception as error: #Si hubo un error en el frame significa que no se cargaron los datos
            print(f"\n Error al cargar detecciones: {error}")
            indice_frame = indice_frame +1
            pbar.update(1)
            continue

        indice_frame = indice_frame + 1
        pbar.update(1)









Generando datos de robots...:  56%|█████▌    | 10816/19407 [06:56<03:58, 36.01Frame/s]


 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00010812.json'

 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00010813.json'

 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00010814.json'


Generando datos de robots...:  65%|██████▍   | 12556/19407 [07:56<03:15, 34.97Frame/s]


 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00012553.json'


Generando datos de robots...:  74%|███████▍  | 14416/19407 [09:11<02:10, 38.23Frame/s]

...

Generando datos de robots...:  77%|███████▋  | 15024/19407 [09:32<03:12, 22.82Frame/s]


 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00015020.json'

 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00015021.json'

 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00015022.json'

 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00015023.json'


Generando datos de robots...:  98%|█████████▊| 18973/19407 [12:06<00:17, 24.70Frame/s]


 Error al cargar detecciones: [Errno 2] No such file or directory: '/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo/frame_00018969.json'


Generando datos de robots...: 100%|█████████▉| 19393/19407 [12:23<00:00, 26.09Frame/s]


## Guardar los datos en el dataframe y exportarlo

In [ ]:

#VERIFICAR QUE TODAS LAS LISTAS TENGAN LA MISMA LONGIDUT, DE OTRA MANERA
#EL DATAFRAME NO SE PODRA GENERAR
print("Len frame: ",len(datos["frame"]))
print("Len tracker_id: ",len(datos["tracker_id"]))
print("Len class_name: ",len(datos["class_name"]))
print("Len x_canon: ",len(datos["x_canon"]))
print("Len y_canon: ",len(datos["y_canon"]))

print("---------------------------------------------")

print("Len area_caja: ",len(datos["area_caja"]))
print("Len area_mascara: ",len(datos["area_mascara"]))
print("Len ancho_caja: ",len(datos["ancho_caja"]))
print("Len alto_caja: ",len(datos["alto_caja"]))

Len frame:  43044
Len tracker_id:  43044
Len class_name:  43044
Len x_canon:  43044
Len y_canon:  43044
---------------------------------------------
Len area_caja:  43044
Len area_mascara:  43044
Len ancho_caja:  43044
Len alto_caja:  43044


In [ ]:
df_robots = pd.DataFrame(datos)

In [ ]:
df_robots.head()

,frame,tracker_id,class_name,x_canon,y_canon,area_caja,area_mascara,ancho_caja,alto_caja
0,1,0,robot,89.750,53.750,18923.0,14931,127.0,149.0
1,1,1,robot,88.125,166.875,16714.0,13286,122.0,137.0
2,1,2,robot,87.875,79.875,16625.0,12901,125.0,133.0
3,2,0,robot,89.875,53.000,19737.0,15447,129.0,153.0
4,2,1,robot,87.875,166.250,17112.0,13751,124.0,138.0


In [ ]:
df_robots.to_csv(RUTA_OBJETIVO,index=False,sep=",",encoding="utf-8-sig")